In [0]:
dbutils.widgets.dropdown("env", "dev", ["dev", "uat", "prod"], "Environment")
env = dbutils.widgets.get("env")
print(f"Running in Environment: {env}")

In [0]:
import os
import json

# Notebook is in src/transformations/, config is in <project>/config/
config_path = os.path.realpath(os.path.join("..", "..", "config", f"{env}.json"))

if not os.path.isfile(config_path):
    raise FileNotFoundError(f"Config for env '{env}' not found at {config_path}")

with open(config_path, "r") as f:
    config = json.load(f)

REQUIRED_KEYS = [
    "CATALOG", "SCHEMA", "KAFKA_TABLE", "CHECKPOINT_VOLUME",
    "SILVER_SCHEMA", "KAFKA_SILVER_TABLE", "SILVER_CHECKPOINT_FOLDER","QUARANTINE_TABLE","QUARANTINE_CHECKPOINT_FOLDER"
]
missing = [k for k in REQUIRED_KEYS if k not in config]
if missing:
    raise KeyError(f"Config '{config_path}' is missing keys: {missing}")

for k in REQUIRED_KEYS:
    print(f"  {k:28s} : {config[k]}")

In [0]:
catalog = config["CATALOG"]

bronze_table_fqn = f"{catalog}.{config['SCHEMA']}.{config['KAFKA_TABLE']}"
silver_table_fqn = f"{catalog}.{config['SILVER_SCHEMA']}.{config['KAFKA_SILVER_TABLE']}"

# Same checkpoint volume as bronze, but a separate folder: one checkpoint per streaming query
silver_checkpoint_path = (
    f"/Volumes/{catalog}/{config['SCHEMA']}/{config['CHECKPOINT_VOLUME']}/{config['SILVER_CHECKPOINT_FOLDER']}"
)

print(f"Bronze table    : {bronze_table_fqn}")
print(f"Silver table    : {silver_table_fqn}")
print(f"Checkpoint path : {silver_checkpoint_path}")

In [0]:
import json
from pyspark.sql.types import *
schema = StructType([
        StructField("transaction_id", StringType()),
        StructField("customer_id", StringType()),
        StructField("card_number", StringType()),
        StructField("merchant_id", StringType()),
        StructField("merchant_name", StringType()),
        StructField("merchant_category", StringType()),
        StructField("amount", DoubleType()),
        StructField("currency", StringType()),
        StructField("transaction_type", StringType()),
        StructField("payment_channel", StringType()),
        StructField("device_id", StringType()),
        StructField("city", StringType()),
        StructField("country", StringType()),
        StructField("transaction_timestamp", TimestampType()),
        StructField("is_international", BooleanType()),
        StructField("status", StringType())
    ])
print(schema.simpleString())

In [0]:
bronze_stream_df = (
    spark.readStream
    .option("maxFilesPerTrigger", 1000)     # cap files per micro-batch (default 1000)
    .option("skipChangeCommits", "true")    # ignore bronze versions created by UPDATE/DELETE/MERGE (e.g. privacy deletes)
    .table(bronze_table_fqn)
)

print(f"Is streaming : {bronze_stream_df.isStreaming}")
bronze_stream_df.printSchema()

In [0]:
from pyspark.sql import functions as F

def transform_bronze_to_silver(df):
    parsed = df.select(
        F.from_json(F.col("value"), schema).alias("txn"),
        F.get_json_object("value", "$.amount").alias("raw_amount"),   # raw value, to detect type mismatches
        F.col("topic").alias("kafka_topic"),
        F.col("partition").alias("kafka_partition"),
        F.col("offset").alias("kafka_offset"),
        F.col("timestamp").alias("kafka_timestamp"),
    )

    cleaned = parsed.select(
        F.trim("txn.transaction_id").alias("transaction_id"),
        F.trim("txn.customer_id").alias("customer_id"),
        F.sha2(F.col("txn.card_number"), 256).alias("card_number_hash"),
        F.right(F.col("txn.card_number"), F.lit(4)).alias("card_last4"),
        F.trim("txn.merchant_id").alias("merchant_id"),
        F.trim("txn.merchant_name").alias("merchant_name"),
        F.initcap(F.trim("txn.merchant_category")).alias("merchant_category"),
        F.col("txn.amount").cast("decimal(18,2)").alias("amount"),
        F.upper(F.trim("txn.currency")).alias("currency"),
        F.upper(F.trim("txn.transaction_type")).alias("transaction_type"),
        F.upper(F.trim("txn.payment_channel")).alias("payment_channel"),
        F.trim("txn.device_id").alias("device_id"),
        F.initcap(F.trim("txn.city")).alias("city"),
        F.initcap(F.trim("txn.country")).alias("country"),
        F.col("txn.transaction_timestamp").alias("transaction_ts"),   # already TimestampType from your schema
        F.coalesce(F.col("txn.is_international"), F.lit(False)).alias("is_international"),
        F.upper(F.trim("txn.status")).alias("status"),
        "raw_amount", "kafka_topic", "kafka_partition", "kafka_offset", "kafka_timestamp",
    )

    enriched = (
        cleaned
        .withColumn("transaction_date", F.to_date("transaction_ts"))
        .withColumn("transaction_hour", F.hour("transaction_ts"))
        .withColumn("silver_processed_ts", F.current_timestamp())
        .withColumn(
            "dq_issue",
            F.when(F.col("transaction_id").isNull(), "missing_transaction_id")
             .when(F.col("customer_id").isNull(), "missing_customer_id")
             .when(F.col("transaction_ts").isNull(), "invalid_transaction_ts")
             .when(F.col("amount").isNull() & F.col("raw_amount").isNotNull(), "amount_type_mismatch")
             .when(F.col("amount").isNull(), "missing_amount")
             .when(F.col("amount") <= 0, "non_positive_amount")
        )
        .drop("raw_amount")
    )
    return enriched

In [0]:
silver_stream_df = transform_bronze_to_silver(bronze_stream_df)
print(f"Is streaming : {silver_stream_df.isStreaming}")
silver_stream_df.printSchema()

In [0]:
test_df = transform_bronze_to_silver(spark.read.table(bronze_table_fqn))

display(test_df.limit(5))

display(
    test_df.groupBy("dq_issue").count()
)

In [0]:
DEDUP_WATERMARK = "1 hour"   # max expected gap between an original and its duplicate

silver_valid_df = (
    silver_stream_df
    .filter(F.col("dq_issue").isNull())
    .withWatermark("kafka_timestamp", DEDUP_WATERMARK)
    .dropDuplicatesWithinWatermark(["transaction_id"])
    .drop("dq_issue")
)

silver_invalid_df = silver_stream_df.filter(F.col("dq_issue").isNotNull())

print(f"Valid stream   : {silver_valid_df.isStreaming}, columns = {len(silver_valid_df.columns)}")
print(f"Invalid stream : {silver_invalid_df.isStreaming}, columns = {len(silver_invalid_df.columns)}")

In [0]:
quarantine_table_fqn = f"{catalog}.{config['SILVER_SCHEMA']}.{config['QUARANTINE_TABLE']}"
quarantine_checkpoint_path = (
    f"/Volumes/{catalog}/{config['SCHEMA']}/{config['CHECKPOINT_VOLUME']}/{config['QUARANTINE_CHECKPOINT_FOLDER']}"
)

# Create an empty quarantine table if it doesn't exist, so readers and alerts never fail on a
# missing table. Appending an empty DataFrame is safe to repeat.
spark.createDataFrame([], silver_invalid_df.schema).write.mode("append").saveAsTable(quarantine_table_fqn)

print(f"Quarantine table      : {quarantine_table_fqn}")
print(f"Quarantine checkpoint : {quarantine_checkpoint_path}")

In [0]:
silver_query = (
    silver_valid_df.writeStream.format("delta")
    .queryName(f"cc_fraud_silver_{env}")
    .outputMode("append")
    .option("checkpointLocation", silver_checkpoint_path)
    .trigger(availableNow=True)
    .toTable(silver_table_fqn)
)

quarantine_query = (
    silver_invalid_df.writeStream.format("delta")
    .queryName(f"cc_fraud_silver_quarantine_{env}")
    .outputMode("append")
    .option("checkpointLocation", quarantine_checkpoint_path)
    .trigger(availableNow=True)
    .toTable(quarantine_table_fqn)
)

# Block until both finish; raises here if either stream fails
silver_query.awaitTermination()
quarantine_query.awaitTermination()

for q in [silver_query, quarantine_query]:
    p = q.lastProgress or {}
    print(f"{q.name:35s} batch={p.get('batchId')} input_rows={p.get('numInputRows')} "
          f"output_rows={p.get('sink', {}).get('numOutputRows')}")

In [0]:
bronze_cnt     = spark.table(bronze_table_fqn).count()
silver_cnt     = spark.table(silver_table_fqn).count()
silver_ids     = spark.table(silver_table_fqn).select("transaction_id").distinct().count()
quarantine_cnt = spark.table(quarantine_table_fqn).count()

print(f"Bronze rows            : {bronze_cnt}")
print(f"Silver rows            : {silver_cnt}")
print(f"Silver distinct txn ids: {silver_ids}")
print(f"Quarantine rows        : {quarantine_cnt}")
print(f"Dropped as duplicates  : {bronze_cnt - silver_cnt - quarantine_cnt}")

In [0]:
%sql
SELECT
  count(*)                                          AS total_rows,
  count_if(length(card_last4) <> 4)                 AS bad_last4,
  count_if(length(card_number_hash) <> 64)          AS bad_hash,
  count_if(amount IS NULL OR amount <= 0)           AS bad_amount,
  count_if(transaction_date <> to_date(transaction_ts)) AS bad_date,
  min(transaction_ts)                               AS min_ts,
  max(transaction_ts)                               AS max_ts
FROM myorg_dev.silver.silver_kafka_fraud